# Tahap 4: Train & Evaluate CBF
Melakukan proses pelatihan model (*training*) `TFIDFRecommender` menggunakan dataset dari Tahap 3. 
Selanjutnya akan dilakukan evaluasi interaktif pencarian rekomendasi berdasar kemiripan resep (*Cosine Similarity*), dilanjutkan dengan Evaluasi LOO (HR@K & MRR).

In [11]:
import sys
import pandas as pd
from pathlib import Path

# Impor Model CBF kita dari folder models/
sys.path.append('.')
from models.tfidf_model import TFIDFRecommender

MODEL_DIR = Path("outputs/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
IN_FILE = Path("outputs/models/cbf_features.pkl")

In [12]:
print("1. Memuat Dataset CBF Features (Tahap 03)...")
df_cbf = pd.read_pickle(IN_FILE)

print(f"Total dataset resep: {df_cbf.shape[0]}")
display(df_cbf[['id', 'cf_idx', 'combined_text']].head(2))

1. Memuat Dataset CBF Features (Tahap 03)...
Total dataset resep: 75017


,id,cf_idx,combined_text
0,40,0,best lemonade sugar lemons rind of fresh water...
1,49,1,chicken breasts lombardi fresh mushroom butter...


In [13]:
print("2. Melatih Model TF-IDF...")

recommender = TFIDFRecommender(max_features=5000)
recommender.fit(df_cbf, text_col='combined_text', id_col='id')

save_path = MODEL_DIR / "tfidf_cbf_model.pkl"
recommender.save(save_path)
print("Pelatihan model telah selesai.")

2. Melatih Model TF-IDF...
Fitting TF-IDF model on 75017 recipes...
TF-IDF Matrix shape: (75017, 5000)
Model saved to outputs\models\tfidf_cbf_model.pkl
Pelatihan model telah selesai.


In [14]:
print("3. Evaluasi Model (inferensi interaktif)")
sample_id = recommender.item_ids[0]
print(f"\nTarget Pencarian Kemiripan Untuk ID Resep: {sample_id}")

log_results = []
similars = recommender.get_similar_items(sample_id, top_k=5)

for i, (sid, score) in enumerate(similars):
    print(f"Peringkat {i+1}. Resep {sid} -> Skor Kemiripan: {score:.5f}")
    log_results.append({
        "target_recipe_id": sample_id,
        "rank": i+1,
        "similar_recipe_id": sid,
        "similarity_score": score
    })

# Simpan hasil log ke CSV di cbf/results/
results_dir = Path("outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "inference_results_cbf.csv"

df_results = pd.DataFrame(log_results)
df_results.to_csv(results_file, index=False)
print(f"\nHasil log interaktif telah disimpan di {results_file}")

3. Evaluasi Model (inferensi interaktif)

Target Pencarian Kemiripan Untuk ID Resep: 40
Peringkat 1. Resep 207946 -> Skor Kemiripan: 0.57183
Peringkat 2. Resep 19336 -> Skor Kemiripan: 0.55430
Peringkat 3. Resep 237876 -> Skor Kemiripan: 0.54830
Peringkat 4. Resep 118354 -> Skor Kemiripan: 0.53971
Peringkat 5. Resep 176608 -> Skor Kemiripan: 0.52715

Hasil log interaktif telah disimpan di outputs\results\inference_results_cbf.csv


In [15]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(str(Path.cwd().parent))
from cf.data_prep import load_cf_splits, build_user_history, prepare_loo_eval, get_matrix_dims
from cf.evaluator import evaluate_loo, metrics_to_dataframe

print("4. Evaluasi LOO (HR@K & MRR) dengan Pendekatan Item-Based CBF...")

# 4.1 Memuat dataset interaksi CF
cf_train, cf_val, cf_test = load_cf_splits()

# Kita buat user history (item apa saja yang disukai/diinteraksikan user di train & val)
user_history = build_user_history(cf_train, cf_val)
n_users, n_items = get_matrix_dims(cf_train, cf_val, cf_test)

# Menyiapkan data Leave-One-Out (Test vs 99 negatives)
loo_data = prepare_loo_eval(cf_test, user_history, n_items, n_negatives=99, seed=42)

# Buat pemetaan cf_idx -> id_resep asli untuk menjembatani evaluasi CF dengan model CBF 
cbf_map = dict(zip(df_cbf['cf_idx'], df_cbf['id']))

# 4.2 Wrapper Class supaya evaluasi CBF bisa dibaca oleh cf.evaluator
class CBFUserRecommender:
    def __init__(self, cbf_model, user_hist, id_map):
        self.cbf = cbf_model
        self.user_history = user_hist
        self.id_map = id_map
        self.mat = self.cbf.tfidf_matrix
        
        # Precompute valid history matrix indices per user
        self.valid_history = {}
        for u, items in self.user_history.items():
            valid_idx = []
            for cf_idx in items:
                real_id = self.id_map.get(cf_idx)
                if real_id is not None and real_id in self.cbf.item_id_to_idx:
                    valid_idx.append(self.cbf.item_id_to_idx[real_id])
            if valid_idx:
                self.valid_history[u] = valid_idx
                
    def score_candidates(self, u: int, candidates: np.ndarray) -> np.ndarray:
        hist_idx = self.valid_history.get(u)
        if not hist_idx:
            # Jika user tidak punya history yang terekam di CBF, beri skor 0
            return np.zeros(len(candidates))
            
        cand_idx = []
        valid_mask = []
        for c in candidates:
            real_id = self.id_map.get(c)
            if real_id is not None and real_id in self.cbf.item_id_to_idx:
                cand_idx.append(self.cbf.item_id_to_idx[real_id])
                valid_mask.append(True)
            else:
                cand_idx.append(0) # dummy value, akan di-masking
                valid_mask.append(False)
                
        # Hitung Similarity Cosine antara History User dan Candidates items
        hist_vecs = self.mat[hist_idx]
        cand_vecs = self.mat[cand_idx]
        sims = cosine_similarity(hist_vecs, cand_vecs)
        
        # Skor akhir kandidat adalah nilai similarity tertinggi dengan daftar riwayat user (Max-Pooling)
        scores = sims.max(axis=0)
        
        # Masking untuk candidate yang tidak tercover
        scores = scores * np.array(valid_mask)
        return scores

# Evaluasi Model
eval_model = CBFUserRecommender(recommender, user_history, cbf_map)
result_metrics = evaluate_loo(eval_model, loo_data, k_values=(5, 10, 20), verbose=False)

# Menggunakan format eksport serupa dengan notebook 02_train_evaluate_cf
results_df = metrics_to_dataframe({"CBF_TFIDF": result_metrics})

print("\nKESIMPULAN EVALUASI MODEL CBF:")
print(results_df.to_string(float_format=lambda x: f"{x:.4f}"))

# Simpan hasil log ke CSV
results_dir = Path("outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "final_results_cbf.csv"

results_df.to_csv(results_file)
print(f"\nExport Artifact Metrics Berhasil -> {results_file}")

4. Evaluasi LOO (HR@K & MRR) dengan Pendekatan Item-Based CBF...
[data_prep] CF LOO split loaded -- Train: 622,730  Val: 19,123  Test: 19,123
[data_prep] Matrix dims -- Users: 19,123  Items: 75,017
[data_prep] LOO users for evaluation: 19,123
[data_prep] LOO eval records ready: 19,123

KESIMPULAN EVALUASI MODEL CBF:
            HR@5  HR@10  HR@20    MRR
Model                                
CBF_TFIDF 0.1431 0.2312 0.3742 0.1137

Export Artifact Metrics Berhasil -> outputs\results\final_results_cbf.csv
